## 1.5: Galaxy Schema


In [0]:
%sql
DROP TABLE IF EXISTS propiedades.gold.fact_consultas;

CREATE TABLE propiedades.gold.fact_consultas (
    consulta_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    zona_id BIGINT,
    fecha_id BIGINT,
    propiedad_url STRING,
    tipo_consulta STRING,
    cantidad_consultas INT DEFAULT 1,
    _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Tabla de hechos de consultas - Galaxy Schema (mockeada)';


-----------------------------------------------
-- inserta Fact 
-----------------------------------------------

INSERT INTO propiedades.gold.fact_consultas 
(
    zona_id, fecha_id, propiedad_url, tipo_consulta, cantidad_consultas
)
SELECT 
    dz.zona_id,
    dtf.fecha_id,
    sp.url as propiedad_url,
    CASE (RANDOM() * 4)::INT
        WHEN 0 THEN 'vista'
        WHEN 1 THEN 'contacto'
        WHEN 2 THEN 'favorito'
        ELSE 'compartir'
    END as tipo_consulta,
    (RANDOM() * 10 + 1)::INT as cantidad_consultas
FROM propiedades.silver.propiedades sp
JOIN propiedades.gold.dim_zona dz    
ON sp.partido = dz.partido 
AND sp.region = dz.region

JOIN propiedades.gold.dim_tiempo dtf 
ON sp.fecha_publicacion = dtf.fecha

WHERE sp.precio_por_m2 < 
        (
            SELECT PERCENTILE_APPROX(precio_por_m2, 0.5) 
            FROM propiedades.silver.propiedades
        )
LIMIT 5000;


In [0]:
%sql
-- Preview: filas cargadas en fact_consultas (Galaxy)
SELECT * 
FROM    propiedades.gold.fact_consultas 
ORDER BY consulta_id LIMIT 20;


In [0]:
%sql
-- Análisis cruzado entre hechos

SELECT 
        dz.partido, dz.region,
        COUNT(DISTINCT fp.row_hash) as total_propiedades,
        COUNT(DISTINCT fc.consulta_id) as total_consultas,
        SUM(fc.cantidad_consultas) as cantidad_total_consultas,
        ROUND(AVG(fp.precio_por_m2), 2) as precio_m2_promedio,
        ROUND(SUM(fc.cantidad_consultas) * 1.0 / NULLIF(COUNT(DISTINCT fp.row_hash), 0), 2) as consultas_por_propiedad

FROM propiedades.gold.dim_zona dz
LEFT JOIN propiedades.gold.fact_propiedades fp 
ON dz.zona_id = fp.zona_id

LEFT JOIN propiedades.gold.fact_consultas fc 
ON dz.zona_id = fc.zona_id

GROUP BY dz.partido, dz.region
HAVING COUNT(DISTINCT fp.row_hash) > 0
ORDER BY consultas_por_propiedad DESC
LIMIT 20;


## 1.6: Comparar esquemas


In [0]:
%sql
-- Star Schema (2 JOINs)

SELECT  dz.partido, 
        dz.region, 
        dt.tipo_operacion, 
        AVG(f.precio_por_m2) as avg_precio_m2

FROM      propiedades.gold.fact_propiedades f
LEFT JOIN propiedades.gold.dim_zona dz 
ON f.zona_id = dz.zona_id

LEFT JOIN propiedades.gold.dim_tipo_operacion dt 
ON f.tipo_operacion_id = dt.tipo_operacion_id

GROUP BY dz.partido, 
         dz.region, 
         dt.tipo_operacion
ORDER BY avg_precio_m2 DESC
LIMIT 10;

In [0]:
%sql
-- Snowflake Schema (4 JOINs)

SELECT  dz.partido, 
        dz.region, 
        dt.tipo_operacion, 
        AVG(f.precio_por_m2) as avg_precio_m2

FROM propiedades.gold.fact_propiedades f

LEFT JOIN propiedades.gold.dim_zona_sf dz 
ON f.zona_id = dz.zona_id
LEFT JOIN propiedades.gold.dim_ciudad_sf dc 
ON dz.ciudad_id = dc.ciudad_id
LEFT JOIN propiedades.gold.dim_provincia_sf dp 
ON dc.provincia_id = dp.provincia_id
LEFT JOIN propiedades.gold.dim_tipo_operacion dt 
ON f.tipo_operacion_id = dt.tipo_operacion_id

GROUP BY dz.partido, dz.region, dt.tipo_operacion
ORDER BY avg_precio_m2 DESC
LIMIT 10;

In [0]:
%sql
-- OBT (0 JOINs)

SELECT  partido, 
        region, 
        tipo_operacion, 
        AVG(precio_por_m2) as avg_precio_m2

FROM propiedades.gold.obt_propiedades_completa
GROUP BY partido, region, tipo_operacion
ORDER BY avg_precio_m2 DESC
LIMIT 10;

In [0]:
%sql
-- Galaxy Schema (mismo que Star — fact_propiedades comparte dimensiones)

SELECT  dz.partido, 
        dz.region, 
        dt.tipo_operacion, 
        AVG(f.precio_por_m2) as avg_precio_m2

FROM propiedades.gold.fact_propiedades f
LEFT JOIN propiedades.gold.dim_zona dz 
ON f.zona_id = dz.zona_id

LEFT JOIN propiedades.gold.dim_tipo_operacion dt 
ON f.tipo_operacion_id = dt.tipo_operacion_id

GROUP BY dz.partido, dz.region, dt.tipo_operacion
ORDER BY avg_precio_m2 DESC
LIMIT 10;